In [1]:
from langchain_openai import ChatOpenAI


#Messages
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage, ToolCall

#Prompts
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

# pydantic
from pydantic import BaseModel, Field
from typing import Annotated, TypedDict, List, Optional

# outputparsers
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser


# chains
from langchain_core.runnables import RunnableBranch, RunnableLambda

# dotenv
from dotenv import load_dotenv

/Users/sriharshamadireddy/Desktop/Personal/Learning/DataScience/genai/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
load_dotenv()
model = ChatOpenAI()

In [31]:
model.invoke("Hi")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EIsD7NAcJeUZAg4n4pOW2HLmzSYGw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0570f-03e6-7840-989c-91230a25ff5d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [32]:
oldMessages = [
    HumanMessage("Hi..! This is human"),
    AIMessage("Hi...! This is AI"),
]

In [36]:
prompt1 = ChatPromptTemplate([
    MessagesPlaceholder(variable_name="oldMessages"),
    ('human', "what is the reponse of {toolCallName} tool call")
])

prompt1= prompt1.invoke({"oldMessages": oldMessages, "toolCallName": "calculator"})
model.invoke(prompt1)

AIMessage(content='The response of a calculator tool call is to perform mathematical calculations based on the input provided by the user. It can add, subtract, multiply, divide, and perform other mathematical operations as needed.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 36, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EIsDZXYCIYvxylSFxB6Rs74f2jD2Q', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0570f-709d-7a91-b030-67b3fbf16f0a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 39, 'total_tokens': 75, 'input_token_details': {'audio': 0, 'cache_

In [62]:
class Person(BaseModel):
    name: str = Field(..., description="name of the person")


modelWithStructuredOutput = model.with_structured_output(Person)
res1 = modelWithStructuredOutput.invoke("I am abcd, who am i")

print(res1.name)


pydanticOutputParser = PydanticOutputParser(pydantic_object=Person)

prompt2= PromptTemplate(
    template= "who am I, {name} \n {format_instructions}",
    input_variables=['name'],
    partial_variables={"format_instructions": pydanticOutputParser.get_format_instructions()}
)

pydanticOutputParser.invoke(model.invoke(prompt2.invoke({'name': 'abcd'})))


# runnable branch
stroutputParser = StrOutputParser()
chainConditional = RunnableBranch(
    (lambda x: x.name == 'abcd', RunnableLambda(lambda x: "Thank you message") | model | stroutputParser),
    (lambda x: x.name == 'def', RunnableLambda(lambda x: "Thank you message") | model | stroutputParser),
    RunnableLambda(lambda _: "DefaultCase")
)

chain1 = prompt2 | model | pydanticOutputParser | chainConditional
chain1.invoke({"name": "abcd"})





abcd


'Thank you so much for your kindness and support. Your thoughtfulness means the world to me and I am truly grateful for all that you do. Your generosity and love have made a positive impact on my life, and I am so blessed to have you in my life. Thank you from the bottom of my heart.'

In [4]:
class A(TypedDict):
    a: Annotated[str, Field(..., description="a field")]